# Particle trajectory animation -- run_2 (1993-2013)

Parameterised (papermill) notebook that renders ONE period of the 1993-2013 Mercator/Parcels
production run as an MP4, in the style of `Amazon/Atlantic/animations/Animation_science_day.ipynb`
(cartopy map, coloured trails per release date, `trail_len`, `fps`, `dpi`).

Because a 5-year block holds ~365 releases x 10061 trajectories = 3.7M trajectories, which cannot be
drawn, a random subsample of `n_particles` trajectories is drawn from every `release_spacing_days`-th
release. Frames advance in **calendar time**, so all releases that are alive at a given date are drawn
simultaneously, each with its own colour (cyclic colormap on release day-of-year).

In [1]:
# --- papermill parameters ---
period_start        = "1993-01-01"   # first release date to include
period_end          = "1997-12-31"   # last release date to include
n_particles         = 400            # randomly chosen trajectories per release
release_spacing_days = 10            # draw a release every N days (stores exist every 5 days)
particle_lifetime_days = 185         # length of each release (740 obs x 6 h)
frame_step_days     = 2              # calendar days between animation frames
trail_len           = 40             # trail length in OBS (6-hourly) steps; 40 obs = 10 days
trail_stride_obs    = 2              # use every Nth obs inside the trail; 2 = 12-hourly vertices
fps                 = 6
dpi                 = 100
out_dir             = "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/animations/video_particles_1993-1997"
out_name            = "particles_1993-1997.mp4"
lon_min, lon_max    = -80.0, -42.0
lat_min, lat_max    = -2.0, 25.0
coast_res           = "50m"
max_frames          = 0              # 0 = all frames; positive N = test render of first N frames
seed                = 20250101
data_root           = "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2"
ffmpeg_path         = ""             # "" = auto-detect (imageio_ffmpeg first, then /sw)

In [2]:
# Parameters
period_start = "1998-01-01"
period_end = "2002-12-31"
n_particles = 400
release_spacing_days = 10
frame_step_days = 2
trail_len = 40
fps = 6
dpi = 100
max_frames = 0
out_dir = "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/animations/video_particles_1998-2002"
out_name = "particles_1998-2002.mp4"


In [3]:
import os, glob, re, time, gc
import numpy as np
import xarray as xr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# ffmpeg: the parcels_3.1.2 env has no ffmpeg on PATH, but imageio_ffmpeg ships a static
# ffmpeg 7.0.2 with libx264. Fall back to the system mambaforge ffmpeg 6.1.1 (also libx264).
# NOTE: /sw/spack-levante/ffmpeg-4.3.2-rmywep/bin/ffmpeg has NO libx264 and must not be used.
if not ffmpeg_path:
    try:
        import imageio_ffmpeg
        ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        ffmpeg_path = "/sw/spack-levante/mambaforge-23.11.0-0-Linux-x86_64-rb7hor/bin/ffmpeg"
if not os.path.exists(ffmpeg_path):
    raise RuntimeError("no usable ffmpeg found: " + ffmpeg_path)
matplotlib.rcParams["animation.ffmpeg_path"] = ffmpeg_path
os.makedirs(out_dir, exist_ok=True)
out_mp4 = os.path.join(out_dir, out_name)
print("output:", out_mp4)
print("ffmpeg:", ffmpeg_path, os.path.exists(ffmpeg_path))

output: /work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/animations/video_particles_1998-2002/particles_1998-2002.mp4
ffmpeg: /work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/imageio_ffmpeg/binaries/ffmpeg-linux-x86_64-v7.0.2 True


## 1. Select the release stores for this period

In [4]:
# The run is split over four seed directories; collect every store and keep the ones whose
# release date falls inside [period_start, period_end].
TRACK_DIRS = ["tracks_v3_11098723", "tracks_v3_45678", "tracks_v3_67891", "tracks_v3_78876"]

pat = re.compile(r"Parcels_run_(\d+)_(\d{4}-\d{2}-\d{2})")
all_stores = []
for d in TRACK_DIRS:
    for p in glob.glob(os.path.join(data_root, d, "Parcels_run_*.zarr")):
        m = pat.search(os.path.basename(p))
        if m:
            all_stores.append((np.datetime64(m.group(2)), p))
all_stores.sort()
print("stores found in total:", len(all_stores))

t0 = np.datetime64(period_start)
t1 = np.datetime64(period_end)
in_period = [(d, p) for d, p in all_stores if t0 <= d <= t1]
print("stores inside period:", len(in_period), in_period[0][0], "..", in_period[-1][0])

# Thin to one release every `release_spacing_days`: walk forward and keep the next store that is
# at least release_spacing_days after the previously kept one.
selected = []
last = None
for d, p in in_period:
    if last is None or (d - last) / np.timedelta64(1, "D") >= release_spacing_days:
        selected.append((d, p))
        last = d
print("releases drawn:", len(selected), "| particles per release:", n_particles,
      "| trajectories drawn in total:", len(selected) * n_particles)

stores found in total: 1534
stores inside period: 366 1998-01-01 .. 2002-12-31
releases drawn: 183 | particles per release: 400 | trajectories drawn in total: 73200


## 2. Load only the subsampled trajectories

Each store is opened, `n_particles` trajectory indices are drawn at random (reproducible, seeded per
release date), `lon`/`lat` are read as `float32` for those indices only, and the store is closed
again. Nothing is ever concatenated, so peak memory stays at
`n_releases x n_particles x 740 x 2 x 4 bytes` (a few hundred MB for a 5-year block).

In [5]:
releases = []
t_load = time.time()
for i, (d, p) in enumerate(selected):
    ds = xr.open_zarr(p)
    ntraj = ds.sizes["trajectory"]
    rng = np.random.default_rng(seed + int(d.astype("datetime64[D]").astype(int)))
    k = min(n_particles, ntraj)
    idx = np.sort(rng.choice(ntraj, size=k, replace=False))
    sub = ds[["lon", "lat"]].isel(trajectory=idx).load()
    releases.append({
        "date": d,
        "t0": ds.time.isel(trajectory=0, obs=0).values.astype("datetime64[ns]"),
        "lon": sub.lon.values.astype("float32"),
        "lat": sub.lat.values.astype("float32"),
        "nobs": int(sub.sizes["obs"]),
    })
    ds.close(); del ds, sub
    if i % 20 == 0 or i == len(selected) - 1:
        gc.collect()
        print(f"  loaded {i+1}/{len(selected)}  {d}  ({time.time()-t_load:.0f}s)", flush=True)
print(f"load finished in {time.time()-t_load:.0f}s")

# Store lengths are not identical across the run (most releases have 740 obs = 185 days, some
# reruns have 925 obs = 231 days), so every release keeps its own obs count.
import collections
print("obs per release:", dict(collections.Counter(r["nobs"] for r in releases)))
mb = sum(r["lon"].nbytes + r["lat"].nbytes for r in releases) / 1e6
print("in-memory size: %.0f MB" % mb)
short = [str(r["date"]) for r in releases if r["nobs"] < 100]
if short:
    print("WARNING: truncated releases (<100 obs):", short)

  loaded 1/183  1998-01-01  (3s)


  loaded 21/183  1998-07-20  (6s)


  loaded 41/183  1999-02-05  (9s)


  loaded 61/183  1999-08-24  (12s)


  loaded 81/183  2000-03-11  (14s)


  loaded 101/183  2000-09-27  (17s)


... (2 outputs left out)

  loaded 161/183  2002-05-20  (29s)


  loaded 181/183  2002-12-06  (32s)


  loaded 183/183  2002-12-26  (32s)


load finished in 32s
obs per release: {740: 175, 925: 8}
in-memory size: 438 MB


## 3. Colours and the calendar-time frame axis

In [6]:
# Colour each release by its day-of-year with a cyclic colormap, so the seasonal cycle of the
# releases is visible and neighbouring releases stay distinguishable.
cmap = plt.get_cmap("twilight")
doy = np.array([(r["date"] - r["date"].astype("datetime64[Y]")) / np.timedelta64(1, "D")
                for r in releases])
for r, dy in zip(releases, doy):
    r["color"] = cmap((dy % 365.0) / 365.0)

dt_obs_h = 6.0                      # obs spacing in hours
frame_step = np.timedelta64(int(frame_step_days * 24), "h")
t_first = min(r["t0"] for r in releases)
t_last = max(r["t0"] + np.timedelta64(int((r["nobs"] - 1) * dt_obs_h), "h") for r in releases)
frame_times = np.arange(t_first, t_last + frame_step, frame_step)
if max_frames:
    frame_times = frame_times[:max_frames]
print("frames:", len(frame_times), frame_times[0], "..", frame_times[-1])
print("video duration: %.1f s at %d fps" % (len(frame_times) / fps, fps))

frames: 1004 1998-01-01T00:00:00.000000000 .. 2003-06-30T00:00:00.000000000
video duration: 167.3 s at 6 fps


## 4. Render

In [7]:
def obs_index(r, t):
    return int(round((t - r["t0"]) / np.timedelta64(int(dt_obs_h * 60), "m")))

proj = ccrs.PlateCarree()
fig = plt.figure(figsize=(12, 8.5), dpi=dpi)
ax = plt.axes(projection=proj)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=proj)
ax.add_feature(cfeature.LAND.with_scale(coast_res), facecolor="#f2f2f2", edgecolor="none", zorder=0)
ax.coastlines(resolution=coast_res, linewidth=0.5, zorder=1)
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="0.7", alpha=0.6, linestyle=":")
gl.top_labels = False; gl.right_labels = False

tail = LineCollection([], linewidths=0.7, alpha=0.65, transform=proj, zorder=2)
ax.add_collection(tail)
head = ax.scatter([], [], s=5, marker="o", linewidths=0, edgecolors="none", transform=proj, zorder=10)

sm = ScalarMappable(norm=Normalize(0, 365), cmap=cmap)
cb = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.02)
cb.set_ticks([15, 105, 197, 288, 350])
cb.set_ticklabels(["Jan", "Apr", "Jul", "Oct", "Dec"])
cb.set_label("release date (day of year)")

title = ax.set_title("")
sub = fig.text(0.5, 0.02,
               "%d releases x %d particles drawn (of ~10061 per release)" % (len(releases), n_particles),
               ha="center", fontsize=9, color="0.35")

L = max(2, int(trail_len))
stride = max(1, int(trail_stride_obs))

writer = FFMpegWriter(fps=fps, codec="libx264", bitrate=-1,
                      extra_args=["-pix_fmt", "yuv420p", "-preset", "medium", "-crf", "23"])

t_render = time.time()
with writer.saving(fig, out_mp4, dpi=dpi):
    for fi, t in enumerate(frame_times):
        segs, seg_cols, hx, hy, hc = [], [], [], [], []
        nlive = 0
        for r in releases:
            k = obs_index(r, t)
            if k < 0 or k >= r["nobs"]:
                continue
            s = max(0, k - (L - 1) * stride)
            idxs = np.arange(k, s - 1, -stride)[::-1]
            if idxs.size >= 2:
                w_lon = r["lon"][:, idxs]; w_lat = r["lat"][:, idxs]
                x0 = w_lon[:, :-1]; x1 = w_lon[:, 1:]
                y0 = w_lat[:, :-1]; y1 = w_lat[:, 1:]
                mm = np.isfinite(x0) & np.isfinite(x1) & np.isfinite(y0) & np.isfinite(y1)
                n = int(mm.sum())
                if n:
                    s4 = np.empty((n, 2, 2), dtype=np.float32)
                    s4[:, 0, 0] = x0[mm]; s4[:, 0, 1] = y0[mm]
                    s4[:, 1, 0] = x1[mm]; s4[:, 1, 1] = y1[mm]
                    segs.append(s4)
                    seg_cols.append(np.repeat(np.asarray(r["color"], dtype=np.float32)[None, :], n, axis=0))
            m = np.isfinite(r["lon"][:, k]) & np.isfinite(r["lat"][:, k])
            nh = int(m.sum())
            if nh:
                hx.append(r["lon"][m, k]); hy.append(r["lat"][m, k])
                hc.append(np.repeat(np.asarray(r["color"], dtype=np.float32)[None, :], nh, axis=0))
                nlive += nh

        if segs:
            tail.set_segments(np.concatenate(segs, axis=0))
            tail.set_color(np.concatenate(seg_cols, axis=0))
        else:
            tail.set_segments([])
        if hx:
            head.set_offsets(np.column_stack([np.concatenate(hx), np.concatenate(hy)]))
            head.set_facecolor(np.concatenate(hc, axis=0))
        else:
            head.set_offsets(np.empty((0, 2), dtype=np.float32))

        title.set_text("%s    live particles: %d" % (np.datetime_as_string(t, unit="D"), nlive))
        writer.grab_frame()
        if fi % 25 == 0 or fi == len(frame_times) - 1:
            el = time.time() - t_render
            print(f"  frame {fi+1}/{len(frame_times)}  live={nlive}  {el:.0f}s "
                  f"({el/max(fi+1,1):.2f} s/frame)", flush=True)

plt.close(fig)
render_s = time.time() - t_render
print("render finished in %.0f s (%.2f s/frame)" % (render_s, render_s / len(frame_times)))

  frame 1/1004  live=400  0s (0.38 s/frame)


  frame 26/1004  live=2400  18s (0.70 s/frame)


  frame 51/1004  live=4400  72s (1.41 s/frame)


  frame 76/1004  live=6400  163s (2.14 s/frame)


  frame 101/1004  live=7600  288s (2.85 s/frame)


  frame 126/1004  live=7600  420s (3.33 s/frame)


... (33 outputs left out)

  frame 976/1004  live=2400  4758s (4.88 s/frame)


  frame 1001/1004  live=400  4785s (4.78 s/frame)


  frame 1004/1004  live=0  4786s (4.77 s/frame)


render finished in 4786 s (4.77 s/frame)


## 5. Summary

In [8]:
size_mb = os.path.getsize(out_mp4) / 1e6
print("file          :", out_mp4)
print("size          : %.1f MB" % size_mb)
print("frames        :", len(frame_times), "at %d fps -> %.1f s" % (fps, len(frame_times) / fps))
print("period        :", period_start, "..", period_end)
print("releases drawn:", len(releases), "(every %d days)" % release_spacing_days)
print("particles/rel :", n_particles, "-> %d trajectories drawn" % (len(releases) * n_particles))
print("render time   : %.0f s" % render_s)

file          : /work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/animations/video_particles_1998-2002/particles_1998-2002.mp4
size          : 36.9 MB
frames        : 1004 at 6 fps -> 167.3 s
period        : 1998-01-01 .. 2002-12-31
releases drawn: 183 (every 10 days)
particles/rel : 400 -> 73200 trajectories drawn
render time   : 4786 s
